# Surface Nets

Surface nets ({cite:t}`gibson1998`) is a dual method: instead of cutting triangles out of
each cell the way marching cubes does, it places one vertex inside
every cell the surface crosses and connects the vertices of
neighboring cells. The vertex sits at the centroid of the points
where the surface crosses the cell's edges. That placement needs no
normals and no solves, so the method has no parameters to tune, and
the mesh comes out smooth and evenly tessellated.


In [1]:
import isoext
from isoext import viewer
from isoext.sdf import CuboidSDF, MandelbulbSDF, get_sdf_normal
from isoext.utils import gaussian_smooth


## Basic Usage

```python
vertices, faces = isoext.surface_nets(grid, level=0.0)
```

- `grid`: A `UniformGrid` or `SparseGrid` with values set
- `level`: The iso-value to extract (default: 0.0)
- `intersection`: Optional precomputed edge crossings from
  `get_intersection`; computed automatically when omitted


In [2]:
grid = isoext.UniformGrid([64, 64, 64])
grid.set_values(grid.get_points().norm(dim=-1) - 0.8)  # Sphere

vertices, faces = isoext.surface_nets(grid)
print(f"{vertices.shape[0]:,} vertices, {faces.shape[0]:,} triangles")


11,954 vertices, 23,904 triangles


## A Smooth Field

Because the vertex placement only uses where the surface crosses the
cell edges, surface nets works well on fields where normals are
unreliable: distance estimators, neural network outputs, or smoothed
data. The Mandelbulb below is a distance estimator, and the mesh
stays clean without any of the tuning dual contouring would need
here:


In [3]:
bulb_grid = isoext.UniformGrid([128] * 3, aabb_min=[-1.2] * 3, aabb_max=[1.2] * 3)
values = MandelbulbSDF(iterations=6)(bulb_grid.get_points())
bulb_grid.set_values(gaussian_smooth(values, sigma=1.0))

v, f = isoext.surface_nets(bulb_grid)
print(f"{v.shape[0]:,} vertices")
viewer.embed(v, f, color="coral")


84,708 vertices


## Versus Dual Contouring

Surface nets and dual contouring build the same mesh connectivity
and differ only in where the vertex goes inside each cell. Dual
contouring solves for the point that best fits the surface normals,
which can reconstruct sharp edges and corners exactly -- provided it
gets good normals, here computed from the SDF gradient. Surface nets
averages the edge crossings, which rounds them off. The cube below
shows the difference. Use dual contouring when the field has sharp
features worth preserving and normals you trust; use surface nets
otherwise. See {doc}`dual_contouring` for the details on normals.


In [4]:
cube_grid = isoext.UniformGrid([64] * 3)
cube = CuboidSDF(size=[1.0, 1.0, 1.0])
cube_grid.set_values(cube(cube_grid.get_points()))

its = isoext.get_intersection(cube_grid)
its.set_normals(get_sdf_normal(cube, its.get_points()))
v, f = isoext.dual_contouring(cube_grid, intersection=its)
print(f"dual_contouring: {v.shape[0]:,} vertices")
viewer.embed(v, f, color="goldenrod", flat_shading=True, height=300)


dual_contouring: 6,146 vertices


In [5]:
v, f = isoext.surface_nets(cube_grid)
print(f"surface_nets:    {v.shape[0]:,} vertices")
viewer.embed(v, f, color="steelblue", flat_shading=True, height=300)


surface_nets:    6,146 vertices


## References

```{bibliography}
:filter: docname in docnames
```
